# Healthcare Urgency Classification Using Natural Language Processing



In [1]:
#Library import 

import pandas as pd
import string
import pandas as pd
!pip install nltk
import re
from nltk.corpus import stopwords
import nltk
from nltk.tokenize import word_tokenize
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.stem import WordNetLemmatizer
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\razan\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\razan\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\razan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\razan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\razan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [2]:
#import dataset 

df = pd.read_csv("dataset.csv", sep=';')
print(df.columns)
df.head()

Index(['text', 'label'], dtype='object')


,text,label
0,severe abdominal pain huardign and rebound ten...,URGENT
1,laceartion to forearm requiring sutures bleedi...,URGENT
2,no acute distress here perop for evaluation su...,ROUTINE
3,followup for stable eczema no new zomplaints. ...,NON-URGENT
4,patient anaphylaxis airway swellign and hypote...,EMERGENCY


In [3]:
# Remove embedded labels from the text column
df["text"] = df["text"].str.replace(r',?(URGENT|ROUTINE|EMERGENCY|NON-URGENT)\s*$', '', regex=True)

# Keep a copy of the original cleaned text for examples
df["original_text"] = df["text"]

# First step- Data Preprocessing 

In [5]:
df["original_text"] = df["text"]

In [6]:
# Example Before Cleaning:
print("Before Cleaning:\n")
print(df["original_text"].iloc[6])


Before Cleaning:

high fever and vomiting persistent for days signs of uehydration


In [7]:
#Converting Text to Lowercase:

df["text"] = df["text"].str.lower()
df["text"].iloc[4]

#Removing Punctuation from Text:

df["text"] = df["text"].apply(lambda x: x.translate(str.maketrans("", "", string.punctuation)))
df["text"].iloc[2]

#Text Cleaning: Lowercasing and Removing Punctuation:

df["cleaned"] =df["text"].str.lower().apply(lambda x: x.translate(str.maketrans("", "", string.punctuation)))

df[["text", "cleaned"]].head(3)


#Example After Cleaning:
print("\nAfter Cleaning:\n")
print(df["text"].iloc[6])

 


After Cleaning:

high fever and vomiting persistent for days signs of uehydration


In [8]:
# Example Before Tokens:
print("Example before tokns:")
print(df['text'].iloc[0])

Example before tokns:
severe abdominal pain huardign and rebound tenderness on exam unclear onset


In [9]:
#Loading English Stopwords:

stop_words = set(stopwords.words('english'))

# Selecting a Text Sample from the Dataset
text = df['text'].iloc[0]

#Text Cleaning, Tokenization, and Stopword Removal:

text_clean = text.lower()
text_clean = re.sub(r'[^a-z\s]', '', text_clean)
tokens = [word for word in text_clean.split() if word not in stop_words]

#Example After Tokns:
print("Example after tokns:")
print(' '.join(tokens))

df.head()

Example after tokns:
severe abdominal pain huardign rebound tenderness exam unclear onset


,text,label,original_text,cleaned
0,severe abdominal pain huardign and rebound ten...,URGENT,severe abdominal pain huardign and rebound ten...,severe abdominal pain huardign and rebound ten...
1,laceartion to forearm requiring sutures bleedi...,URGENT,laceartion to forearm requiring sutures bleedi...,laceartion to forearm requiring sutures bleedi...
2,no acute distress here perop for evaluation su...,ROUTINE,no acute distress here perop for evaluation su...,no acute distress here perop for evaluation su...
3,followup for stable eczema no new zomplaints m...,NON-URGENT,followup for stable eczema no new zomplaints. ...,followup for stable eczema no new zomplaints m...
4,patient anaphylaxis airway swellign and hypote...,EMERGENCY,patient anaphylaxis airway swellign and hypote...,patient anaphylaxis airway swellign and hypote...


In [10]:
#Example Before Stopwords Removal:

print("Before Stopwords Removal:\n")
print(df["text"].iloc[0])

Before Stopwords Removal:

severe abdominal pain huardign and rebound tenderness on exam unclear onset


In [11]:
#Removing Stopwords

def remove_stopwords(text):

    words = word_tokenize(text)

    filtered_words = [word for word in words if word.lower() not in stop_words]

    return " ".join(filtered_words)

df.columns

df["clean_text"] = df["text"].apply(remove_stopwords)

#Example After Stopwords Removal:
 
print("\nAfter Stopwords Removal:\n")
print(df["clean_text"].iloc[0])


After Stopwords Removal:

severe abdominal pain huardign rebound tenderness exam unclear onset


In [12]:
#Example Before Lemmatization:

print("Before Lemmatization:\n")
print(df["clean_text"].iloc[6])


Before Lemmatization:

high fever vomiting persistent days signs uehydration


In [13]:
#Applying Lemmatization to Normalize Words:

lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    
    words = word_tokenize(text)
    
    lemmas = [lemmatizer.lemmatize(word) for word in words]
    
    return " ".join(lemmas)

df["lemmatized_text"] = df["clean_text"].apply(lemmatize_text)

#Example After Lemmatization:

print("\nAfter Lemmatization:\n")
print(df["lemmatized_text"].iloc[6])


After Lemmatization:

high fever vomiting persistent day sign uehydration


# Second step- Feature Extraction

In [15]:
#Feature Extraction using TF-IDF

import pandas as pd

data = pd.read_csv("dataset.csv", sep=';')

texts = df["lemmatized_text"]

from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=1000, ngram_range=(1,2))

X = vectorizer.fit_transform(texts)

print("Shape of TF-IDF Matrix:")
print(X.shape)

Shape of TF-IDF Matrix:
(500, 1000)


In [16]:
#Feature Inspection 

print("Number of features:", len(vectorizer.get_feature_names_out()))
print("Sample features:", vectorizer.get_feature_names_out()[:10])

Number of features: 1000
Sample features: ['abdfmen' 'abdomen' 'abdomen active' 'abdomen atcive' 'abdominal'
 'abdominal pain' 'abnormal' 'abnormal denies' 'abnormal difficult'
 'abnormal history']


In [17]:
#Example 
print("Original Text:")
print(texts.iloc[0])
print("\nTF-IDF Representation:")
print(X[0].toarray())

Original Text:
severe abdominal pain huardign rebound tenderness exam unclear onset

TF-IDF Representation:
[[0.         0.         0.         0.         0.29242253 0.30369014
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.       

# The third step- Model Implementation

In [19]:
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Define features and labels
texts = df["lemmatized_text"]
y = df["label"]

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(
    texts, y, test_size=0.2, random_state=42, stratify=y
)

# Convert text to TF-IDF
vectorizer = TfidfVectorizer(max_features=1000, ngram_range=(1,2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Train the SVM model
model = SVC(kernel='linear')
model.fit(X_train_tfidf, y_train)

# Predict on test set
y_pred = model.predict(X_test_tfidf)

# Evaluate the model
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))

Accuracy: 1.0

Classification Report:

              precision    recall  f1-score   support

   EMERGENCY       1.00      1.00      1.00        25
  NON-URGENT       1.00      1.00      1.00        25
     ROUTINE       1.00      1.00      1.00        25
      URGENT       1.00      1.00      1.00        25

    accuracy                           1.00       100
   macro avg       1.00      1.00      1.00       100
weighted avg       1.00      1.00      1.00       100


Confusion Matrix:

[[25  0  0  0]
 [ 0 25  0  0]
 [ 0  0 25  0]
 [ 0  0  0 25]]
